In [1]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import blackjax
from blackjax.ns.utils import finalise, log_weights, sample, ess
import tqdm

from lumen import (
    make_params, make_cosmology, make_log_likelihood,
    load_sed_Fnu, observed_sed, MODEL_1A, kpc,
)

In [2]:
cosmo = make_cosmology(71.0, 0.27, 0.73)
data_knot = load_sed_Fnu("mydata.csv", unit="mJy")
data_ext = load_sed_Fnu("mydata_ext.csv", unit="mJy")

print(f"Knot: {len(data_knot)} points, upper limits: {data_knot.has_upper_limits}")
print(f"Ext:  {len(data_ext)} points, upper limits: {data_ext.has_upper_limits}")

Knot: 3 points, upper limits: False
Ext:  3 points, upper limits: True


In [3]:
FIXED_COMMON = dict(
    q_ratio=1.0, p=2.7,
    gamma_min=1e2, gamma_max=1e6,
    z=2.5, eta_e=0.1, model=int(MODEL_1A),
)

In [4]:
FIXED_KNOT = {**FIXED_COMMON, "Rj": 3 * kpc, "l": 3 * kpc}
FIXED_EXT  = {**FIXED_COMMON, "Rj": 3 * kpc, "l": 80 * kpc}

In [5]:
SAMPLED = {
    "G0":      (2.0,   30.0),
    "theta":   (5.0,   25.0),
    "Lj":      (1e46,  1e50),
}

In [6]:
param_names = list(SAMPLED.keys())
ndim = len(param_names)
lo = jnp.array([SAMPLED[k][0] for k in param_names])
hi = jnp.array([SAMPLED[k][1] for k in param_names])

In [7]:
def vec_to_params_knot(x):
    kw = dict(FIXED_KNOT)
    for i, name in enumerate(param_names):
        kw[name] = x[i]
    return make_params(**kw)


def vec_to_params_ext(x):
    kw = dict(FIXED_EXT)
    for i, name in enumerate(param_names):
        kw[name] = x[i]
    return make_params(**kw)

def logprior_fn(x):
    in_bounds = jnp.all((x >= lo) & (x <= hi))
    log_vol = jnp.sum(jnp.log(hi - lo))
    return jnp.where(in_bounds, -log_vol, -jnp.inf)

In [8]:
_loglik_knot = make_log_likelihood(data_knot, cosmo=cosmo, nx=64, ngamma=64)
_loglik_ext  = make_log_likelihood(data_ext,  cosmo=cosmo, nx=64, ngamma=64)


def loglikelihood_fn(x):
    params_knot = vec_to_params_knot(x)
    params_ext  = vec_to_params_ext(x)
    return _loglik_knot(params_knot) + _loglik_ext(params_ext)

In [9]:
n_live = 500
num_delete = 50
num_inner_steps = 5 * ndim

algo = blackjax.nss(
    logprior_fn=logprior_fn,
    loglikelihood_fn=loglikelihood_fn,
    num_delete=num_delete,
    num_inner_steps=num_inner_steps,
)

rng_key = jax.random.PRNGKey(42)
rng_key, init_key = jax.random.split(rng_key)
initial_population = jax.random.uniform(
    init_key, (n_live, ndim), minval=lo, maxval=hi,
)

state = algo.init(initial_population)
step = jax.jit(algo.step)

dead = []

print(f"\nRunning joint nested sampling: {ndim} dims, {n_live} live points")
with tqdm.tqdm(desc="Dead points", unit=" dead points") as pbar:
    while not state.integrator.logZ_live - state.integrator.logZ < -3:
        rng_key, subkey = jax.random.split(rng_key)
        state, dead_info = step(subkey, state)
        dead.append(dead_info)
        pbar.update(num_delete)


Running joint nested sampling: 3 dims, 500 live points


Dead points: 0 dead points [00:04, ? dead points/s]


KeyboardInterrupt: 

In [ ]:
rng_key, wkey, skey = jax.random.split(rng_key, 3)
final = finalise(state, dead)
ns_ess = ess(skey, final)
log_w = log_weights(wkey, final, shape=100)
logzs = jax.scipy.special.logsumexp(log_w, axis=0)
posterior = sample(skey, final, shape=n_live)

print(f"\nlogZ = {logzs.mean():.2f} ± {logzs.std():.2f}")
print(f"ESS  = {int(ns_ess)}")
print()
for i, name in enumerate(param_names):
    vals = posterior.position[:, i]
    print(f"  {name:12s}: {vals.mean():.4g} ± {vals.std():.4g}")

In [ ]:
import anesthetic
import matplotlib.pyplot as plt

ns_samples = anesthetic.NestedSamples(
    data=final.particles.position,
    columns=param_names,
    logL=final.particles.loglikelihood,
    logL_birth=final.particles.loglikelihood_birth,
)

prior_plot = ns_samples.set_beta(0.0).plot_2d(
    param_names, label="prior"
)
ns_samples.plot_2d(prior_plot, label="posterior")
prior_plot.iloc[-1, 0].legend(
    bbox_to_anchor=(len(prior_plot), len(prior_plot)),
    loc="lower right",
)

In [ ]:
import matplotlib.pyplot as plt

nu_plot = jnp.array(10 ** np.arange(7, 26, 0.05))

median_vec = jnp.median(posterior.position, axis=0)
p_knot = vec_to_params_knot(median_vec)
p_ext  = vec_to_params_ext(median_vec)
flux_knot = observed_sed(nu_plot, p_knot, cosmo, nx=64, ngamma=64)
flux_ext  = observed_sed(nu_plot, p_ext,  cosmo, nx=64, ngamma=64)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(nu_plot, flux_knot, "-",  color="C0", lw=1.5, label="knot (median)")
ax.plot(nu_plot, flux_ext,  "--", color="C1", lw=1.5, label="extended (median)")
ax.plot(nu_plot, flux_knot + flux_ext, "-", color="0.4", lw=1, alpha=0.5, label="total")

# Knot data
ax.errorbar(
    np.array(data_knot.frequencies), np.array(data_knot.fluxes),
    yerr=np.array(data_knot.errors),
    fmt="o", color="C0", ms=6, capsize=3, zorder=10, label="knot data",
)

# Extended jet data — distinguish detections from upper limits
ext_det = data_ext.detections()
ext_ul  = data_ext.upper_limits()

if len(ext_det):
    ax.errorbar(
        np.array(ext_det.frequencies), np.array(ext_det.fluxes),
        yerr=np.array(ext_det.errors),
        fmt="s", color="C1", ms=6, capsize=3, zorder=10, label="ext data",
    )
if len(ext_ul):
    ax.errorbar(
        np.array(ext_ul.frequencies), np.array(ext_ul.fluxes),
        yerr=0.3 * np.array(ext_ul.fluxes),  # arrow length
        fmt="v", color="C1", ms=6, uplims=True, zorder=10, label="ext upper limit",
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\nu$ [Hz]", fontsize=13)
ax.set_ylabel(r"$\nu F_\nu$ [erg/s/cm$^2$]", fontsize=13)
ax.set_xlim(1e7, 1e26)
ax.set_ylim(1e-20,1e-10)
ax.legend(fontsize=10)
fig.tight_layout()
# fig.savefig("ns_sed_joint.pdf", dpi=150)
# print("SED plot saved to ns_sed_joint.pdf")